In [ ]:
import numpy as np

# Coeficientes de aquecimento para as 6 AZs
C = np.array([42.0, 35.0, 58.0, 30.0, 50.0, 65.0])
LIMITE_TEMP = 75.0
PENALIDADE = 10000.0

def calcular_fitness(pesos):
    """
    Calcula a temperatura de cada AZ e aplica penalidade se ultrapassar o limite.
    O objetivo é minimizar a temperatura média ponderada.
    """
    temperaturas = pesos * C
    temp_media = np.mean(temperaturas)

    # Função de penalidade externa caso alguma AZ passe de 75 °C
    if np.any(temperaturas > LIMITE_TEMP):
        return temp_media + PENALIDADE

    return temp_media

class Particula:
    def __init__(self, dim):
        # Inicializa a posição aleatória e normaliza para a soma ser 1.0
        pos = np.random.rand(dim)
        self.posicao = pos / np.sum(pos)

        # Inicializa a velocidade
        self.velocidade = np.random.uniform(-0.1, 0.1, dim)

        # Histórico da melhor posição pessoal (P_best)
        self.p_best = np.copy(self.posicao)
        self.p_best_fitness = calcular_fitness(self.posicao)

def executar_pso(tamanho_populacao, iteracoes=100, dim=6):
    w = 0.5  # Inércia
    c1 = 1.5 # Coeficiente cognitivo
    c2 = 1.5 # Coeficiente social

    enxame = [Particula(dim) for _ in range(tamanho_populacao)]

    # Encontrar o G_best inicial
    g_best = np.copy(enxame[0].posicao)
    g_best_fitness = float('inf')

    for p in enxame:
        if p.p_best_fitness < g_best_fitness:
            g_best_fitness = p.p_best_fitness
            g_best = np.copy(p.p_best)

    historico_fitness = []

    # Ciclo de otimização
    for _ in range(iteracoes):
        for p in enxame:
            # Atualização da velocidade
            r1, r2 = np.random.rand(), np.random.rand()
            p.velocidade = (w * p.velocidade +
                            c1 * r1 * (p.p_best - p.posicao) +
                            c2 * r2 * (g_best - p.posicao))

            # Atualização da posição
            p.posicao = p.posicao + p.velocidade

            # Operador de normalização e garantia de pesos não-negativos
            p.posicao = np.clip(p.posicao, 0.0001, 1.0)
            p.posicao = p.posicao / np.sum(p.posicao)

            # Avaliação do fitness
            fitness_atual = calcular_fitness(p.posicao)

            # Atualizar P_best
            if fitness_atual < p.p_best_fitness:
                p.p_best_fitness = fitness_atual
                p.p_best = np.copy(p.posicao)

                # Atualizar G_best
                if fitness_atual < g_best_fitness:
                    g_best_fitness = fitness_atual
                    g_best = np.copy(p.posicao)

        historico_fitness.append(g_best_fitness)

    return g_best, g_best_fitness, historico_fitness

# ---------------------------------------------------------
# Testes com diferentes tamanhos de população
# ---------------------------------------------------------
tamanhos_pop = [10, 30, 50]

print("RESULTADOS DO LAB 01 - PSO CONTÍNUO\n" + "="*50)

for pop in tamanhos_pop:
    melhor_W, melhor_fit, evolucao = executar_pso(tamanho_populacao=pop)

    print(f"População: {pop} partículas")
    print(f"Melhor Distribuição (W): {np.round(melhor_W, 4)}")
    print(f"Soma dos pesos (Validação): {np.sum(melhor_W):.4f}")
    print(f"Fitness Final (Temp. Média): {melhor_fit:.2f} °C")
    print(f"Temperaturas por AZ: {np.round(melhor_W * C, 2)} °C")
    print("-" * 50)

RESULTADOS DO LAB 01 - PSO CONTÍNUO
População: 10 partículas
Melhor Distribuição (W): [1.000e-04 9.995e-01 1.000e-04 1.000e-04 1.000e-04 1.000e-04]
Soma dos pesos (Validação): 1.0000
Fitness Final (Temp. Média): 5.83 °C
Temperaturas por AZ: [0.000e+00 3.498e+01 1.000e-02 0.000e+00 0.000e+00 1.000e-02] °C
--------------------------------------------------
População: 30 partículas
Melhor Distribuição (W): [1.000e-04 1.000e-04 1.000e-04 9.995e-01 1.000e-04 1.000e-04]
Soma dos pesos (Validação): 1.0000
Fitness Final (Temp. Média): 5.00 °C
Temperaturas por AZ: [0.000e+00 0.000e+00 1.000e-02 2.999e+01 0.000e+00 1.000e-02] °C
--------------------------------------------------
População: 50 partículas
Melhor Distribuição (W): [1.000e-04 1.000e-04 1.000e-04 9.995e-01 1.000e-04 1.000e-04]
Soma dos pesos (Validação): 1.0000
Fitness Final (Temp. Média): 5.00 °C
Temperaturas por AZ: [0.000e+00 0.000e+00 1.000e-02 2.999e+01 0.000e+00 1.000e-02] °C
--------------------------------------------------
